# 01 - Data Exploration 
------------------------------------------------------------------------------------------------------------------
*Start: 02-05-2026, Latest change: 02-05-2026*

The purpose of this notebook is to initially explore the data that can be found on NASAs FIRMS API. By doing this, the specific question and project focus will be developed, which is the fundamental basis of this SDS210 geospatial project. 

All required packages for this code can be found in the *enviroment.yml* file.

------------------------------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------------------------------
### 1.1 Preliminaries and Data Aquisition
Loading required libraries, importing data via API request. Important to note: the MAP_KEY needs to be an accepted, unique token which can be requested via the following link of NASA FIRMS: https://firms.modaps.eosdis.nasa.gov/api/map_key/

Theory used from SDS210 public Gitbook: *Week 5 - Libraries; 04 - Using Web APIs*

In [17]:
# 1.1.0 Importing libraries
import requests
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO

# 1.1.1) Specify API URL and make a request
## specify different subvariables of API
#### generate accepted token with earthdata
#### accepted token needed as ID for Web API
MAP_KEY = "c1637686346099a509207bbf06a29650"

DATA = "VIIRS_SNPP_NRT"
AREA = "world"
DAY_RANGE = 1

## building URL based on those subvariables (makes it more specific)
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{DATA}/{AREA}/{DAY_RANGE}"

# 1.1.2 Requesting data
response = requests.get(api_url)

# 1.1.3 Check whether request worked by checking status code and print first few lines
if response.status_code == 200:
    print("Request successful!")
    print(response.text[:300]) # successful if column headers are visible

Request successful!
latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
67.13725,56.78384,336.42,0.43,0.46,2026-05-05,18,N,VIIRS,n,2.0NRT,270.43,3.54,D
67.13853,56.78001,336.36,0.43,0.46,2026-05-05,18,N,VIIRS,n,2.0NRT,272.08,3.49,D
55.58586,51.92562


------------------------------------------------------------------------------------------------------------------
### 1.2 Reading Data &  Exploration, Cleaning, Visualization
The Data is currently in a **CSV format**, we need to convert it into a more useful format for analyzing it, best to use **Pandas**. AI suggested to use the function **StringIO** (treats a string like a file) to be able to convert it into a Pandas data frame. *This command could also later be incorporated into a function itself (including the API request) to make it more automated.*

Inspect the Pandas DataFrame (general structure, Column names, NaNs, Data Types). Additionally, research on the FIRMS website for the meaning of the different attributes: Go to *area*, scroll down to the links for the attribute tables of each satellite (https://firms.modaps.eosdis.nasa.gov/content/descriptions/FIRMS_VIIRS_Firehotspots.html?utm_source=chatgpt.com). 

**Location:** Lat/Lon (WGS84) // 
**Fire intensity:** bright_ti4 (Brighness temperature in K on I4 band), bright_ti5 (same as before but on I5 band, used for comparison/background temperature), frp (Fire Radiative Power, as a proxy for fire size & intensity) //
**Pixel geometry:** scan (pixel size across track, km), track (pixel size along track, km) = how large the pixel is //
**Time:** acq_date (date of observation UTC, needs to be converted to a timestamp), acq_time (time of observation in HHMM format), daynight //
**Satellite info:** satellite, instruments (here VIIRS = sensor) //
**Detection quality:**: confidence (either low, nominal, high = as strings) //

(W6, W7 for help?)

Quick analysis of value distributions, time patterns, spatial patterns, possible relationships. (plotting with Folium W8)

In [32]:
# 1.2.1 convert data from CSV format to a pandas dataframe
df = pd.read_csv(StringIO(response.text))

# 1.2.2 inspect the datas colnames, dtypes, NaNs, etc 
df.info() # 14 cols, lat & lon in the first two
df.isna().sum() # 0 NaN values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9249 entries, 0 to 9248
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   latitude    9249 non-null   float64
 1   longitude   9249 non-null   float64
 2   bright_ti4  9249 non-null   float64
 3   scan        9249 non-null   float64
 4   track       9249 non-null   float64
 5   acq_date    9249 non-null   object 
 6   acq_time    9249 non-null   int64  
 7   satellite   9249 non-null   object 
 8   instrument  9249 non-null   object 
 9   confidence  9249 non-null   object 
 10  version     9249 non-null   object 
 11  bright_ti5  9249 non-null   float64
 12  frp         9249 non-null   float64
 13  daynight    9249 non-null   object 
dtypes: float64(7), int64(1), object(6)
memory usage: 1011.7+ KB


latitude      0
longitude     0
bright_ti4    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_ti5    0
frp           0
daynight      0
dtype: int64

In [34]:
# inspect first 5 rows of data
df.head()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,67.13725,56.78384,336.42,0.43,0.46,2026-05-05,18,N,VIIRS,n,2.0NRT,270.43,3.54,D
1,67.13853,56.78001,336.36,0.43,0.46,2026-05-05,18,N,VIIRS,n,2.0NRT,272.08,3.49,D
2,55.58586,51.92562,307.00,0.49,0.65,2026-05-05,20,N,VIIRS,n,2.0NRT,279.27,0.72,N
3,55.58601,51.92078,316.59,0.49,0.65,2026-05-05,20,N,VIIRS,n,2.0NRT,279.15,2.24,N
4,55.58842,51.91881,297.68,0.49,0.65,2026-05-05,20,N,VIIRS,n,2.0NRT,279.30,0.72,N


In [ ]:
# 1.2.3 convert dtypes, change colnames, get rid of NaNs (data cleaning)

# 1.2.4 convert to a geopandas object and visualize 

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,67.13725,56.78384,336.42,0.43,0.46,2026-05-05,18,N,VIIRS,n,2.0NRT,270.43,3.54,D
1,67.13853,56.78001,336.36,0.43,0.46,2026-05-05,18,N,VIIRS,n,2.0NRT,272.08,3.49,D
2,55.58586,51.92562,307.00,0.49,0.65,2026-05-05,20,N,VIIRS,n,2.0NRT,279.27,0.72,N
3,55.58601,51.92078,316.59,0.49,0.65,2026-05-05,20,N,VIIRS,n,2.0NRT,279.15,2.24,N
4,55.58842,51.91881,297.68,0.49,0.65,2026-05-05,20,N,VIIRS,n,2.0NRT,279.30,0.72,N
